# Tracing a Calcite Beam Displacer

A surface with a tensor material on either side uses the anisotropic
interaction model (`interaction_type="anisotropic"`). The rays of a
polarized optic then carry their wave vector k beside the ray direction:
they move along the Poynting vector, the optical path of a step Δr is
Re(k) · Δr, and the PRT matrix of each surface is that of the mode the ray
follows.

One trace follows one mode at each anisotropic surface: `"o"` or `"e"` in a
uniaxial crystal, `"slow"` or `"fast"` in any crystal, and `"T"` (the whole
transmitted field) into an isotropic medium. This example traces both modes
of a calcite beam displacer and compares the displacement, the optical path
and the power with closed forms.

In [1]:
import numpy as np

from optiland.materials import IdealMaterial, UniaxialMaterial
from optiland.optic import Optic
from optiland.rays import PolarizationState

n_o, n_e = 1.6583434042, 1.4861300612  # calcite at 589.3 nm (Ghosh 1999)
wavelength = 0.5893
thickness = 10.0  # mm

The optic axis lies at 45° in the x–z plane. The input is linearly
polarized at 30° from x: x is the e field of this cut, y the o field.

In [2]:
calcite = UniaxialMaterial(
    IdealMaterial(n_o), IdealMaterial(n_e), optic_axis=(1.0, 0.0, 1.0)
)

displacer = Optic()
displacer.surfaces.add(index=0, radius=np.inf, thickness=np.inf)
displacer.surfaces.add(
    index=1, thickness=thickness, material=calcite, is_stop=True,
    interaction_type="anisotropic",
)  # fmt: skip
displacer.surfaces.add(index=2, thickness=5.0, interaction_type="anisotropic")
displacer.surfaces.add(index=3)
displacer.set_aperture(aperture_type="EPD", value=2.0)
displacer.fields.set_type(field_type="angle")
displacer.fields.add(y=0.0)
displacer.wavelengths.add(value=wavelength, is_primary=True)

azimuth = np.radians(30.0)
displacer.updater.set_polarization(
    PolarizationState(
        is_polarized=True, Ex=np.cos(azimuth), Ey=np.sin(azimuth),
        phase_x=0.0, phase_y=0.0,
    )
)  # fmt: skip

Trace the chief ray once in each mode of the crystal. The exit face goes
into air, so its mode is `"T"` (the default). `update_intensity` applies
the input polarization state: the ray power is |P E|² times the flux factor
of the path.

In [3]:
traced = {}
for mode in ("o", "e"):
    displacer.surfaces[1].interaction_model.mode = mode
    rays = displacer.trace_generic(
        Hx=0.0, Hy=0.0, Px=0.0, Py=0.0, wavelength=wavelength
    )
    rays.update_intensity(displacer.polarization_state)
    traced[mode] = rays
    print(
        f"{mode}: x = {float(rays.x[0]):+.9f} mm   L, M, N = "
        f"{float(rays.L[0]):+.6f}, {float(rays.M[0]):+.6f}, {float(rays.N[0]):+.6f}"
        f"   power = {float(rays.i[0]):.9f}   OPD = {float(rays.opd[0]):.6f} mm"
    )

o: x = +0.000000000 mm   L, M, N = +0.000000, +0.000000, +1.000000   power = 0.220274689   OPD = 23.583434 mm
e: x = -1.092064213 mm   L, M, N = +0.000000, +0.000000, +1.000000   power = 0.678951904   OPD = 22.651754 mm


## Closed forms

* The e ray walks off by ρ with tan ρ = (n² / 2)(1/n_e² − 1/n_o²) sin 2θ,
  1/n² = cos²θ / n_o² + sin²θ / n_e², θ = 45°, so it leaves the plate
  displaced by −d tan ρ along x (toward the optic axis sense of this cut).
* Both rays leave parallel to the input.
* The power of each mode is the input fraction (cos² or sin² of the azimuth)
  times the transmittance 4n / (1 + n)² of each face at normal incidence,
  with n_o for the o wave and n(45°) for the e wave.
* The optical path in the crystal is Re(k) · Δr = n d for both waves: the e
  ray is longer by 1/cos ρ, but its wave vector makes the angle ρ with it.

In [4]:
t = np.radians(45.0)
n_45 = 1.0 / np.sqrt(np.cos(t) ** 2 / n_o**2 + np.sin(t) ** 2 / n_e**2)
tan_rho = n_45**2 / 2 * (1 / n_e**2 - 1 / n_o**2) * np.sin(2 * t)


def face(n):
    return 4 * n / (1 + n) ** 2


expected = {
    "o": (0.0, np.sin(azimuth) ** 2 * face(n_o) ** 2),
    "e": (-thickness * tan_rho, np.cos(azimuth) ** 2 * face(n_45) ** 2),
}
for mode, (x, power) in expected.items():
    rays = traced[mode]
    print(
        f"{mode}: |x - closed form| = {abs(float(rays.x[0]) - x):.1e} mm,"
        f" |power - closed form| = {abs(float(rays.i[0]) - power):.1e}"
    )

opd_o = float(traced["o"].opd[0])
opd_e = float(traced["e"].opd[0])
print(
    f"OPD(o) - OPD(e) = {opd_o - opd_e:.9f} mm;"
    f" (n_o - n(45°)) d = {(n_o - n_45) * thickness:.9f} mm"
)

o: |x - closed form| = 0.0e+00 mm, |power - closed form| = 2.8e-17
e: |x - closed form| = 2.2e-16 mm, |power - closed form| = 3.3e-16
OPD(o) - OPD(e) = 0.931680287 mm; (n_o - n(45°)) d = 0.931680287 mm


The ray records of the surfaces show the two paths through the plate.

In [5]:
for mode in ("o", "e"):
    displacer.surfaces[1].interaction_model.mode = mode
    displacer.trace_generic(Hx=0.0, Hy=0.0, Px=0.0, Py=0.0, wavelength=wavelength)
    points = [(float(s.x[0]), float(s.z[0])) for s in displacer.surfaces][1:]
    print(mode, "x, z on surfaces 1 to 3:", np.round(points, 6).tolist())

o x, z on surfaces 1 to 3: [[0.0, 0.0], [0.0, 10.0], [0.0, 15.0]]
e x, z on surfaces 1 to 3: [[0.0, 0.0], [-1.092064, 10.0], [-1.092064, 15.0]]
